# Homework 2: Teaching Step by Step

This notebook is meant to feel like a guided lab, not a summary sheet.

For each homework question, the structure is:

1. Quote the question exactly.
2. Check the relevant file or data.
3. Explain what the evidence means.
4. State the answer.

The goal is to make every claim traceable to something we actually inspect in the data.

## 0. Setup

Before answering any question, we need to locate the homework files and the downloaded sequencing data.

This notebook assumes the large data files are in:

```bash
~/compgenomicsws_hw2
```

If your files are elsewhere, change `DATA_DIR` in the next cell.

In [1]:
from pathlib import Path
import csv
import gzip
import statistics
import zipfile
import pandas as pd
import plotly.express as px
from IPython import get_ipython
from IPython.display import display

DATA_DIR = Path.home() / "compgenomicsws_hw2"
ASSIGNMENT_DIR = Path.cwd()
if not (ASSIGNMENT_DIR / "Metadata_18.csv").exists():
    candidate = ASSIGNMENT_DIR / "Lesson3[IntroNGS]+Assignment2"
    if candidate.exists():
        ASSIGNMENT_DIR = candidate

METADATA = ASSIGNMENT_DIR / "Metadata_18.csv"
HG38 = DATA_DIR / "hg38.fa.gz"
FASTQC_DIR = DATA_DIR / "fastqc"
FASTQ_FILES = {
    "normal_R1": DATA_DIR / "ERR4833597_1.fastq.gz",
    "normal_R2": DATA_DIR / "ERR4833597_2.fastq.gz",
    "tumor_R1": DATA_DIR / "ERR4833621_1.fastq.gz",
    "tumor_R2": DATA_DIR / "ERR4833621_2.fastq.gz",
}

def show_plot(fig):
    ip = get_ipython()
    if ip and ip.__class__.__name__ == "ZMQInteractiveShell":
        fig.show(renderer="plotly_mimetype")
    else:
        print(f"Created plot: {fig.layout.title.text}")

print(f"Assignment directory: {ASSIGNMENT_DIR}")
print(f"Metadata file exists: {METADATA.exists()} -> {METADATA}")
for label, path in FASTQ_FILES.items():
    print(f"{label:10s} exists={path.exists()} path={path}")
print(f"hg38 exists: {HG38.exists()} -> {HG38}")
print(f"FastQC directory exists: {FASTQC_DIR.exists()} -> {FASTQC_DIR}")

Assignment directory: /mnt/c/Users/orgrd/workspace/repos/CompGenomicsWS/Lesson3[IntroNGS]+Assignment2
Metadata file exists: True -> /mnt/c/Users/orgrd/workspace/repos/CompGenomicsWS/Lesson3[IntroNGS]+Assignment2/Metadata_18.csv
normal_R1  exists=True path=/home/orgr/compgenomicsws_hw2/ERR4833597_1.fastq.gz
normal_R2  exists=True path=/home/orgr/compgenomicsws_hw2/ERR4833597_2.fastq.gz
tumor_R1   exists=True path=/home/orgr/compgenomicsws_hw2/ERR4833621_1.fastq.gz
tumor_R2   exists=True path=/home/orgr/compgenomicsws_hw2/ERR4833621_2.fastq.gz
hg38 exists: True -> /home/orgr/compgenomicsws_hw2/hg38.fa.gz
FastQC directory exists: True -> /home/orgr/compgenomicsws_hw2/fastqc


## 1. Metadata Questions

### **Q1:** The file [Metadata_18.csv](Metadata_18.csv) contains additional meta information about the files we are downloading. What does 'neoplasm' and 'normal tissue adjacent to neoplasm' mean? </br>

### **Q2:** The 23rd column (LIBRARY_STRATEGY) is marked as WXS. What is Whole Exome Sequencing (WXS/WES)? What would be the fastq file sizes if we did Whole Genome Sequencing (WGS) instead?</br></br>

We should not guess from memory alone. First we inspect the metadata rows that belong to the downloaded FASTQ files.

In [2]:
with METADATA.open(newline="") as fh:
    reader = csv.reader(fh)
    header = next(reader)
    rows = list(reader)

interesting_columns = [
    "Characteristics[individual]",
    "Characteristics[organism part]",
    "Characteristics[sampling site]",
    "Characteristics[disease]",
    "Characteristics[genotype]",
    "Comment[LIBRARY_LAYOUT]",
    "Comment[LIBRARY_STRATEGY]",
    "Scan Name",
]
indices = [header.index(col) for col in interesting_columns]
metadata_records = [{col: row[idx] for col, idx in zip(interesting_columns, indices)} for row in rows]

metadata_df = pd.DataFrame(metadata_records)
display(metadata_df)

,Characteristics[individual],Characteristics[organism part],Characteristics[sampling site],Characteristics[disease],Characteristics[genotype],Comment[LIBRARY_LAYOUT],Comment[LIBRARY_STRATEGY],Scan Name
0,AD0699_18,uterine cervix,normal tissue adjacent to neoplasm,cervical cancer,germline genotype,PAIRED,WXS,AD0699_18N_R1.fastq.gz
1,AD0699_18,uterine cervix,normal tissue adjacent to neoplasm,cervical cancer,germline genotype,PAIRED,WXS,AD0699_18N_R2.fastq.gz
2,AD0729_18,uterine cervix,neoplasm,cervical cancer,somatic genotype,PAIRED,WXS,AD0729_18T_R1.fastq.gz
3,AD0729_18,uterine cervix,neoplasm,cervical cancer,somatic genotype,PAIRED,WXS,AD0729_18T_R2.fastq.gz


### Step-by-step reasoning for Q1

Look closely at the `Characteristics[sampling site]` column:

- two rows say `normal tissue adjacent to neoplasm`
- two rows say `neoplasm`

These labels are telling us what kind of tissue was sampled.

Now connect that with the other columns:

- the `normal tissue adjacent to neoplasm` rows are also marked `germline genotype`
- the `neoplasm` rows are also marked `somatic genotype`

That tells us the homework is comparing:

- a tumor sample
- a non-tumor sample from the same patient, used as the comparison/control

So here, `neoplasm` means the tumor tissue, and `normal tissue adjacent to neoplasm` means nearby tissue from the same patient that is not labeled as tumor.

For a computer-science analogy: the normal sample is the baseline version, and the tumor sample is the modified version we want to compare against it.

### Answer to Q1

`neoplasm` means the tumor sample.

`normal tissue adjacent to neoplasm` means nearby non-tumor tissue from the same patient. In this homework it is the matched normal/control sample used for comparison.

### Step-by-step reasoning for Q2

Now look at the `Comment[LIBRARY_STRATEGY]` column. All four rows say `WXS`.

`WXS` means **Whole Exome Sequencing**.

Why is that important? Because the exome is only the protein-coding part of the genome, which is much smaller than the whole human genome.

That already helps explain why these FASTQ files are not enormous. If this were **Whole Genome Sequencing (WGS)**, we would expect much more sequence data, because the experiment would cover all human DNA, not just the exome.

### Answer to Q2

`WXS`/`WES` means whole-exome sequencing. It targets the protein-coding parts of genes, not the whole genome.

If this were whole-genome sequencing (`WGS`), the FASTQ files would usually be much larger, often tens of GB per compressed FASTQ file, because the whole genome is much larger than the exome.

## 2. FASTQ Inspection

### **Q3:** What is the difference between the header of the first read in R1 and the header of the first read in R2?</br>

### **Q4:** What is the read length? How many reads are there per file?</br></br>

The README tells us to start by simply looking at the files. So we will do exactly that.

In [3]:
def first_fastq_record(path):
    with gzip.open(path, "rt") as fh:
        return [next(fh).rstrip() for _ in range(4)]

for label in ["normal_R1", "normal_R2", "tumor_R1", "tumor_R2"]:
    print("=" * 80)
    print(label)
    print("\n".join(first_fastq_record(FASTQ_FILES[label])))

normal_R1
@ERR4833597.1 HISEQ:451:C78DHACXX:7:1101:10000:14561/1
GGCGGCGGCGGCGGCGGCGGCGGCGGCGGCCGGGGCTGGTGGGGGAGGGGGGGTGGCCCCTGATCCTCCGGTAGCGAAGGGCTAGGGGCACACGCCCAGC
+
BBBBFBF<FF7B########################################################################################
normal_R2
@ERR4833597.1 HISEQ:451:C78DHACXX:7:1101:10000:14561/2
GCCCTCCCTGCTCCCGGGCGGCTCTGGAGGAGCTCGGTGTTTGCCTCTAGCCCTTCGCTACCGGAGGATCAGGTTCCTCCCCTCCTCCCCCTCCAGCTCC
+
BBBFFBF<FBFFFFIF7FIFIF7FBB<0<<FBBBFBBB<'7<B<B<<B<770<BBB7<BBBBB<0000B<BBB0<<0<0707<7''007'777B<'0<BB
tumor_R1
@ERR4833621.1 HISEQ:451:C78DHACXX:1:1101:10000:16167/1
CCCACAGCAGCCCAGTGAAGAGTCCTGTTATCTTTTCGGTTTGACAGATGAGGAGATGGAGGCCAGAGAGGATGGGCCACCGGCCAAGGTCATTGTCCTG
+
BBBFFFFFFFFFFIIFIFIIIIFFIFIFFFIFFIIIIIIFIIIIIIIIIIIFIIIFFIII<FFFFFFFBFFFFFFB<BBBFBFBFFF<B0<BBFFFFFBB
tumor_R2
@ERR4833621.1 HISEQ:451:C78DHACXX:1:1101:10000:16167/2
GGGCAGTGACTGAATCCCCAAGAAAGTAGAGCCATCAAGGTCAGGGCTCTGTGGGCACAGGGTTGGGATTTAAATCCCAGCTCCACAGGACAATGACCTT
+
BBBFFFFFFFFFFIIIIIIFIIFIIIB

### Step-by-step reasoning for Q3

A FASTQ record always has four lines:

1. header
2. DNA sequence
3. `+`
4. quality string

For the normal sample, compare the first headers:

- `@ERR4833597.1 HISEQ:451:C78DHACXX:7:1101:10000:14561/1`
- `@ERR4833597.1 HISEQ:451:C78DHACXX:7:1101:10000:14561/2`

Everything matches except the last part:

- `/1` in `R1`
- `/2` in `R2`

That means the two files contain the paired reads from the same original DNA fragment.

### Answer to Q3

The headers are the same except for the mate suffix:

- `R1` ends with `/1`
- `R2` ends with `/2`

So they are the two paired reads from the same fragment.

In [4]:
def fastq_stats(path):
    line_count = 0
    first_read_length = None
    with gzip.open(path, "rt") as fh:
        for line_count, line in enumerate(fh, start=1):
            if line_count == 2:
                first_read_length = len(line.rstrip())
    return {
        "read_length": first_read_length,
        "read_count": line_count // 4,
    }

stats = {label: fastq_stats(path) for label, path in FASTQ_FILES.items()}
read_stats_df = pd.DataFrame([
    {
        "file": label,
        "read_length": result["read_length"],
        "read_count": result["read_count"],
        "sample": "normal" if label.startswith("normal") else "tumor",
        "mate": "R1" if label.endswith("R1") else "R2",
    }
    for label, result in stats.items()
])
display(read_stats_df)

fig = px.bar(
    read_stats_df,
    x="file",
    y="read_count",
    color="sample",
    pattern_shape="mate",
    text="read_count",
    title="Read counts per FASTQ file",
)
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
show_plot(fig)

,file,read_length,read_count,sample,mate
0,normal_R1,100,28927775,normal,R1
1,normal_R2,100,28927775,normal,R2
2,tumor_R1,100,36383853,tumor,R1
3,tumor_R2,100,36383853,tumor,R2


### Step-by-step reasoning for Q4

There are two separate things to measure:

- **read length** = length of the DNA sequence line
- **number of reads** = total FASTQ lines divided by 4, because each read takes four lines

The table shows that every file has read length `100`.

The same table also shows that:

- the two normal files have the same read count
- the two tumor files have the same read count

That is exactly what we expect for paired-end data: `R1` and `R2` should have matching counts.

### Answer to Q4

The read length is `100 bp` in all four files.

Read counts:

- `ERR4833597_1.fastq.gz`: `28,927,775`
- `ERR4833597_2.fastq.gz`: `28,927,775`
- `ERR4833621_1.fastq.gz`: `36,383,853`
- `ERR4833621_2.fastq.gz`: `36,383,853`

## 3. FastQC Questions

### **Q5:** Check the "Per base sequence quality". Do the overall base qualities look ok? Do you notice any strange behavior?</br>

### **Q6:** Does the "Per base sequence content" behave as expected?</br>

Here the important rule is: do not just say "pass" or "fail". We want to inspect the pattern and explain what it means.

In [5]:
def read_fastqc_summary(zip_path):
    with zipfile.ZipFile(zip_path) as zf:
        summary_name = next(name for name in zf.namelist() if name.endswith("summary.txt"))
        text = zf.read(summary_name).decode()
    return [line.split("	") for line in text.strip().splitlines()]

summary_rows = []
for zip_path in sorted(FASTQC_DIR.glob("*_fastqc.zip")):
    for status, module, filename in read_fastqc_summary(zip_path):
        summary_rows.append({
            "file": zip_path.name.replace("_fastqc.zip", ""),
            "module": module,
            "status": status,
        })

fastqc_summary_df = pd.DataFrame(summary_rows)
display(
    fastqc_summary_df[
        fastqc_summary_df["module"].isin([
            "Per base sequence quality",
            "Per base sequence content",
            "Per tile sequence quality",
            "Per sequence GC content",
        ])
    ].pivot(index="file", columns="module", values="status")
)

module,Per base sequence content,Per base sequence quality,Per sequence GC content,Per tile sequence quality
file,,,,
ERR4833597_1,PASS,PASS,WARN,PASS
ERR4833597_2,PASS,PASS,WARN,PASS
ERR4833621_1,PASS,PASS,WARN,WARN
ERR4833621_2,PASS,PASS,WARN,WARN


In [6]:
def extract_fastqc_module(zip_path, module_name):
    with zipfile.ZipFile(zip_path) as zf:
        data_name = next(name for name in zf.namelist() if name.endswith("fastqc_data.txt"))
        lines = zf.read(data_name).decode().splitlines()
    capture = False
    module_lines = []
    for line in lines:
        if line.startswith(f">>{module_name}"):
            capture = True
        if capture:
            module_lines.append(line)
        if capture and line == ">>END_MODULE":
            break
    return module_lines


def module_table(zip_path, module_name):
    lines = extract_fastqc_module(zip_path, module_name)
    header = None
    rows = []
    for line in lines:
        if line.startswith(">>"):
            continue
        if line.startswith("#"):
            header = line[1:].split("	")
            continue
        rows.append(line.split("	"))
    return header, rows


def parse_position_range(value):
    if "-" in value:
        start, end = value.split("-", 1)
        return (float(start) + float(end)) / 2
    return float(value)

fastqc_zip_files = sorted(FASTQC_DIR.glob("*_fastqc.zip"))
quality_rows = []
for zip_path in fastqc_zip_files:
    header, rows = module_table(zip_path, "Per base sequence quality")
    base_i = header.index("Base")
    mean_i = header.index("Mean")
    for row in rows:
        quality_rows.append({
            "file": zip_path.name.replace("_fastqc.zip", ""),
            "position": parse_position_range(row[base_i]),
            "mean_quality": float(row[mean_i]),
        })
quality_df = pd.DataFrame(quality_rows)

display(
    quality_df.sort_values(["file", "mean_quality", "position"]).groupby("file", as_index=False).first()
)

fig = px.line(
    quality_df,
    x="position",
    y="mean_quality",
    color="file",
    title="FastQC mean quality by read position",
)
show_plot(fig)

,file,position,mean_quality
0,ERR4833597_1,76.5,31.975724
1,ERR4833597_2,100.0,31.196168
2,ERR4833621_1,76.5,31.149380
3,ERR4833621_2,100.0,30.691401


### Step-by-step reasoning for Q5

First, the summary table says `Per base sequence quality` is `PASS` for all four files.

But the plot is more informative than the single word `PASS`.

What do we see?

- the lines stay high for most positions, which means most bases have good quality
- the quality drops toward the end of the read, especially in `R2`
- there are visible dips at specific positions instead of a perfectly flat line

So the correct reading is not "perfect data". It is:

- **overall good quality**
- **with a few noticeable dips and the usual end-of-read drop**

That is why it is reasonable to answer that the qualities look OK, while still mentioning the strange behavior.

### Answer to Q5

The overall base qualities look good. All four files pass `Per base sequence quality`.

The strange behavior is that quality is not perfectly flat:

- it drops toward the end of the read, especially in `R2`
- there are visible dips at some positions

So the data look good overall, but the quality profile is not perfectly uniform.

In [7]:
representative_zip = FASTQC_DIR / "ERR4833597_1_fastqc.zip"
if not representative_zip.exists():
    representative_zip = fastqc_zip_files[0]

header, rows = module_table(representative_zip, "Per base sequence content")
base_i = header.index("Base")
composition_rows = []
for row in rows:
    position = parse_position_range(row[base_i])
    for base in ["A", "C", "G", "T"]:
        composition_rows.append({
            "position": position,
            "base": base,
            "percent": float(row[header.index(base)]),
        })
composition_df = pd.DataFrame(composition_rows)

display(composition_df[composition_df["position"] <= 15].head(20))

fig = px.line(
    composition_df,
    x="position",
    y="percent",
    color="base",
    title=f"Base composition by position: {representative_zip.name}",
)
show_plot(fig)

,position,base,percent
0,1.0,A,27.463988
1,1.0,C,23.436370
2,1.0,G,22.614569
3,1.0,T,26.485074
4,2.0,A,27.017303
5,2.0,C,23.782408
6,2.0,G,23.541967
7,2.0,T,25.658322
8,3.0,A,26.305840
9,3.0,C,23.946812


### Step-by-step reasoning for Q6

The FastQC summary says `Per base sequence content` is `PASS`.

Now look at the plot.

If the lines were perfectly flat from the beginning, that would look very clean. But here the first part of the read is a bit biased: the four base percentages do not start out balanced.

The key point is what happens next:

- after the first roughly `10-15` bases, the lines become much more stable

So the data do **not** look perfectly flat at the beginning, but they settle down quickly. That is why the result still passes and is considered acceptable.

### Answer to Q6

Yes, broadly it behaves as expected.

The first roughly `10-15` bases show some bias, but after that the base composition becomes more stable. So it is not perfectly flat at the start, but the overall pattern is still acceptable and passes FastQC.

## 4. Reference Genome Questions

### **Q7:** What is hg38?</br>

### **Q8:** How many sequences are present in the human reference genome you just downloaded?</br>

### **Q9:** Just by looking at the sequence names can you understand what sequences they are?</br>

### **Q10:** How many nucleotides are there in chromosome 1? How many in chromosome 20?

This section is where your question matters a lot: if we say something about `hg38`, we should show where it comes from in the actual downloaded file.

In [8]:
def fasta_lengths(path):
    lengths = {}
    current_name = None
    current_length = 0
    with gzip.open(path, "rt") as fh:
        for line in fh:
            if line.startswith(">"):
                if current_name is not None:
                    lengths[current_name] = current_length
                current_name = line[1:].split()[0]
                current_length = 0
            else:
                current_length += len(line.strip())
    if current_name is not None:
        lengths[current_name] = current_length
    return lengths

lengths = fasta_lengths(HG38)
names = list(lengths)
print(f"File inspected: {HG38}")
print(f"Number of sequences: {len(lengths)}")
print("First 10 sequence names:")
for name in names[:10]:
    print(name)
print("\nLast 10 sequence names:")
for name in names[-10:]:
    print(name)

File inspected: /home/orgr/compgenomicsws_hw2/hg38.fa.gz
Number of sequences: 455
First 10 sequence names:
chr1
chr10
chr11
chr11_KI270721v1_random
chr12
chr13
chr14
chr14_GL000009v2_random
chr14_GL000225v1_random
chr14_KI270722v1_random

Last 10 sequence names:
chrUn_KI270755v1
chrUn_KI270756v1
chrUn_KI270757v1
chrUn_GL000214v1
chrUn_KI270742v1
chrUn_GL000216v2
chrUn_GL000218v1
chrX
chrY
chrY_KI270740v1_random


### Step-by-step reasoning for Q7

How do we know `hg38` is a human reference genome name?

- the downloaded file itself is named `hg38.fa.gz`
- inside the file, the sequence names look like human chromosome names: `chr1`, `chr2`, ..., `chrX`, `chrY`

So this is not just a random label from the internet. The actual data file contains the chromosome naming scheme used by the UCSC human reference.

That is why we say:

- `hg38` = the UCSC name
- `GRCh38` = the formal assembly name
- the file we downloaded is that reference in FASTA format

### Answer to Q7

`hg38` is the UCSC name for the human reference genome assembly `GRCh38`. In this homework, it is the FASTA file we downloaded and inspected as the standard human reference.

### Step-by-step reasoning for Q8

We do not guess the number of sequences. We count them.

In a FASTA file, each new sequence starts with a header line beginning with `>`.

The parsing code above counted the sequences and stored their lengths in the dictionary `lengths`.

So the answer comes directly from the data in `hg38.fa.gz`, not from outside knowledge.

### Answer to Q8

The downloaded `hg38.fa.gz` file contains `455` sequences.

In [9]:
main_chromosomes = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]
all_lengths_df = pd.DataFrame([
    {"sequence": name, "length": length} for name, length in lengths.items()
])

def classify_sequence(name):
    if name in set(main_chromosomes):
        return "main chromosome"
    if name.startswith("chrUn_"):
        return "unplaced contig"
    if "_random" in name:
        return "random/unlocalized scaffold"
    if "_alt" in name:
        return "alternate contig"
    return "other auxiliary sequence"

all_lengths_df["sequence_type"] = all_lengths_df["sequence"].map(classify_sequence)
sequence_type_counts_df = all_lengths_df.groupby("sequence_type", as_index=False).size().rename(columns={"size": "count"})
display(sequence_type_counts_df.sort_values("count", ascending=False))

fig = px.bar(
    sequence_type_counts_df.sort_values("count", ascending=False),
    x="sequence_type",
    y="count",
    text="count",
    title="How many sequences of each type are in hg38?",
)
fig.update_traces(textposition="outside")
show_plot(fig)

,sequence_type,count
0,alternate contig,261
4,unplaced contig,127
3,random/unlocalized scaffold,42
1,main chromosome,24
2,other auxiliary sequence,1


### Step-by-step reasoning for Q9

Now we use the actual sequence names to understand what kinds of sequences are in the file.

From the names themselves:

- `chr1`-`chr22`, `chrX`, `chrY` are the main chromosomes
- `chrUn_...` means sequences that are not placed on one specific chromosome position
- names with `_random` are extra scaffold sequences linked to a chromosome
- accession-like names such as `KI...` and `GL...` are additional reference pieces

The count plot helps here: the file contains a small number of main chromosomes and many extra helper/reference sequences. That is why the total sequence count is much larger than 24.

### Answer to Q9

Yes, at a high level.

The sequence names show that the file contains:

- the main human chromosomes
- unplaced contigs
- random/unlocalized scaffolds
- other extra reference sequences

In [10]:
print(f"chr1:  {lengths['chr1']:,}")
print(f"chr20: {lengths['chr20']:,}")

main_lengths_df = pd.DataFrame({
    "sequence": main_chromosomes,
    "length": [lengths[name] for name in main_chromosomes],
})
fig = px.bar(
    main_lengths_df,
    x="sequence",
    y="length",
    title="Main hg38 chromosome lengths",
)
show_plot(fig)

chr1:  248,956,422
chr20: 64,444,167


### Step-by-step reasoning for Q10

Again, we do not look this up elsewhere. We use the parsed `lengths` dictionary that came from the downloaded FASTA file.

So when we say how many nucleotides are in `chr1` and `chr20`, that answer comes directly from the sequence lengths in this exact `hg38.fa.gz` file.

### Answer to Q10

- `chr1` has `248,956,422` nucleotides.
- `chr20` has `64,444,167` nucleotides.

## 5. Final Teaching Note

The pattern of this homework is very common in bioinformatics:

- inspect the metadata
- inspect the raw text-based file format
- use a standard QC tool
- parse the reference file directly
- only then write the biological interpretation

That is why this notebook keeps asking: "what in the data proves the statement we are making?"